In [3]:
pip install requests

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install requests beautifulsoup4 pandas openpyxl lxml pandas

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install requests beautifulsoup4

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install requests pandas

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install requests openpyxl

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [9]:
pip install requests pandas


Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

BASE_URL = "https://books.toscrape.com/"
print ("Librerias cargadas correctamente")

Librerias cargadas correctamente


1. DESCARGAR EL HTML

In [3]:
response = requests.get(BASE_URL, timeout=10)
print ("Codigo de estado: ", response.status_code)
print ("Longitud del HTML: ", len(response.text))

#vistazo rapido a los primeros 500 caracteres del HTML
print (response.text[:500])

Codigo de estado:  200
Longitud del HTML:  51294
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" /


## Parsear HTML con BeautifulSoup

In [5]:
soup = BeautifulSoup(response.text, "html.parser")

#BUscamos todos los elementos <article class="product">

libros_html = soup.select("article.product_pod")
print (f"Se encontraron {len(libros_html)} libros en esta pagina")

#vemos el primero
print (libros_html[0].prettify()[:600])

Se encontraron 20 libros en esta pagina
<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_pric


## 3. Extraer los datos<

In [11]:
def extraer_rating(clases_css):
    """ El rating viene como una clase CSS: 'star-rating Three' -> 3"""
    mapa = {"One":1, "Two":2, "Three":3, "Four":4, "Five":5}
    for clase in clases_css:
        if clase in mapa:
            return mapa[clase]
        return None

libros = []

for item in libros_html:
    titulo = item.h3.a["title"]
    precio_texto = item.select_one("p.price_color").text.strip()
    precio = float(precio_texto.replace("£", "").replace("Â", ""))
    disponibilidad = item.select_one("p.instock.availability").text.strip()
    rating_clases = item.select_one("p.star-rating")["class"]
    rating = extraer_rating(rating_clases)
    enlace_relativo = item.h3.a["href"]

    libros.append({
        "titulo": titulo,
        "precio_gbp": precio,
        "disponibilidad": disponibilidad,
        "rating": rating,
        "enlace": BASE_URL + enlace_relativo
    })

df_pagina1 = pd.DataFrame(libros)
df_pagina1.head(20)

,titulo,precio_gbp,disponibilidad,rating,enlace
0,A Light in the Attic,51.77,In stock,None,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,In stock,None,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,In stock,None,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,In stock,None,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,In stock,None,https://books.toscrape.com/catalogue/sapiens-a...
5,The Requiem Red,22.65,In stock,None,https://books.toscrape.com/catalogue/the-requi...
6,The Dirty Little Secrets of Getting Your Dream...,33.34,In stock,None,https://books.toscrape.com/catalogue/the-dirty...
7,The Coming Woman: A Novel Based on the Life of...,17.93,In stock,None,https://books.toscrape.com/catalogue/the-comin...
8,The Boys in the Boat: Nine Americans and Their...,22.60,In stock,None,https://books.toscrape.com/catalogue/the-boys-...
9,The Black Maria,52.15,In stock,None,https://books.toscrape.com/catalogue/the-black...


In [12]:
df_pagina1.to_csv("libros_toscrape.csv", index=False, encoding="utf-8-sig")

df_pagina1.to_excel("libros_toscrape.xlsx", index=False)

print("Archivos guardados: libros_toscrape.csv y libros_toscrape.xlsx")

Archivos guardados: libros_toscrape.csv y libros_toscrape.xlsx


## 4. Recorrer varias paginas